
# Benchmark 4 — internal-index reconstruction for `kagome_order_diagnosis_v3`

This notebook tests the **new layer** that reconstructs a leading patch-space mode into an internal-index tensor:

- particle-hole: $\Phi_{ln}(p;Q)$
- particle-particle: $\Delta_{mn}(p)$

The point of this benchmark is **not** to simulate a full FRG flow.  
Instead, it checks that the diagnosis layer can:

1. reuse the coarse recognizer from `form_factor.py` / `order_recognition.py`,
2. reconstruct internal sublattice-pair structure,
3. recognize simple paper-level Kagome templates when the synthetic data are built to match them,
4. keep a novel mode open as `unclassified`.


In [1]:

import json
import numpy as np
import pandas as pd

from kagome_order_diagnosis import KagomeOrderDiagnoser


In [2]:

class FakePatch:
    def __init__(self, k, eigvec):
        self.k = np.asarray(k, float)
        self.eigvec = np.asarray(eigvec, complex)
        self.orbital_weight = np.abs(self.eigvec) ** 2

class FakePatchSet:
    def __init__(self, ks, eigvecs):
        self.patches = [FakePatch(k, u) for k, u in zip(ks, eigvecs)]

class FakeKernel:
    def __init__(self, name, Q, matrix, partner):
        self.name = name
        self.Q = np.asarray(Q, float)
        self.matrix = np.asarray(matrix, complex)
        self.col_partner_patches = np.asarray(partner, int)

    def eig(self, sort_by="abs"):
        vals, vecs = np.linalg.eig(self.matrix)
        order = np.argsort(-np.abs(vals) if sort_by == "abs" else -np.real(vals))
        return vals[order], vecs[:, order]

def circle_ks(N=24):
    th = np.linspace(0, 2 * np.pi, N, endpoint=False)
    return np.c_[np.cos(th), np.sin(th)], th

def projector(vecs):
    V = np.column_stack([np.asarray(v, complex) for v in vecs])
    return V @ V.conj().T



## Synthetic construction logic

Each kernel is a projector
\[
K = \sum_a |v_a\rangle\langle v_a|
\]
so that the intended leading eigenspace is known exactly.

The patch Bloch vectors are also chosen by hand to make the reconstructed tensor
pick out a desired sublattice pair:

- PI / FM: diagonal, same-sublattice
- cBO / sBO: off-diagonal B–C
- f-SC: off-diagonal A–B
- novel pp d-wave: d-wave-like superconducting mode that is **not** one of the PRL paper's five labels


In [3]:

N = 24
ks, theta = circle_ks(N)

v_d1 = np.cos(2 * theta)
v_d2 = np.sin(2 * theta)
v_s  = np.ones(N)

Q1 = np.array([-0.5, -np.sqrt(3.0) / 2.0])
v_bo = np.sin(ks @ Q1)
v_f  = np.sin(1.5 * ks[:, 0] + 0.5 * np.sqrt(3.0) * ks[:, 1])


In [4]:

# 1) PI: same-sublattice diagonal d-wave subspace
patchset_PI = FakePatchSet(ks, np.tile([1, 0, 0], (N, 1)).astype(complex))
kernel_PI = FakeKernel("ph_charge", [0, 0], projector([v_d1, v_d2]), np.arange(N))

# 2) cBO / 3) sBO: finite-Q off-diagonal B-C one-Q component
eigvecs_BC = np.vstack([
    np.tile([0, 1, 0], (N // 2, 1)),
    np.tile([0, 0, 1], (N - N // 2, 1)),
]).astype(complex)
partner_half_shift = np.array([(i + N // 2) % N for i in range(N)])
patchset_BC = FakePatchSet(ks, eigvecs_BC)
kernel_cBO = FakeKernel("ph_charge", Q1, projector([v_bo]), partner_half_shift)
kernel_sBO = FakeKernel("ph_spin",   Q1, projector([v_bo]), partner_half_shift)

# 4) f-SC: triplet-like A-B off-diagonal pairing
eigvecs_AB = np.vstack([
    np.tile([1, 0, 0], (N // 2, 1)),
    np.tile([0, 1, 0], (N - N // 2, 1)),
]).astype(complex)
patchset_AB = FakePatchSet(ks, eigvecs_AB)
kernel_fSC = FakeKernel("pp_triplet_sz0", [0, 0], projector([v_f]), partner_half_shift)

# 5) FM: Q=0 spin-like constant diagonal order
kernel_FM = FakeKernel("ph_spin", [0, 0], projector([v_s]), np.arange(N))

# 6) Novel pp d-wave singlet: physically reasonable, but not one of the five PRL labels
patchset_novel = patchset_PI
kernel_novel = FakeKernel("pp_singlet_sz0", [0, 0], projector([v_d1, v_d2]), np.arange(N))


In [5]:

diagnosers = {
    "PI": KagomeOrderDiagnoser(patchset_PI),
    "cBO": KagomeOrderDiagnoser(patchset_BC),
    "sBO": KagomeOrderDiagnoser(patchset_BC),
    "fSC": KagomeOrderDiagnoser(patchset_AB),
    "FM": KagomeOrderDiagnoser(patchset_PI),
    "novel_pp_dwave": KagomeOrderDiagnoser(patchset_novel),
}

kernels = {
    "PI": kernel_PI,
    "cBO": kernel_cBO,
    "sBO": kernel_sBO,
    "fSC": kernel_fSC,
    "FM": kernel_FM,
    "novel_pp_dwave": kernel_novel,
}

results = {name: diagnosers[name].diagnose_kernel(kernels[name]) for name in kernels}
summary_rows = []
for name, res in results.items():
    s = res.summary_dict()
    row = {
        "case": name,
        "paper_label": s["paper_label"],
        "recognition_status": s["recognition_status"],
        "coarse_label": s["coarse_label"],
        "top_template_name": s["top_template_name"],
        "top_template_score": s["top_template_score"],
        "dominant_pair": s["internal_mode"]["dominant_pair"],
        "same_weight": s["internal_mode"]["same_sublattice_weight"],
        "inter_weight": s["internal_mode"]["inter_sublattice_weight"],
    }
    summary_rows.append(row)

pd.DataFrame(summary_rows)


,case,paper_label,recognition_status,coarse_label,top_template_name,top_template_score,dominant_pair,same_weight,inter_weight
0,PI,PI,recognized,d,PI_dx2y2,0.330267,"(0, 0)",1.0,0.0
1,cBO,cBO,recognized,finiteQ_cos_sin,BO_finiteQ,0.500000,"(1, 2)",0.0,1.0
2,sBO,sBO,recognized,finiteQ_cos_sin,BO_finiteQ,0.500000,"(1, 2)",0.0,1.0
3,fSC,f-SC,recognized,pp_odd,fSC_AB,1.000000,"(0, 1)",0.0,1.0
4,FM,FM,recognized,s,FM_constant,0.333333,"(0, 0)",1.0,0.0
5,novel_pp_dwave,unclassified,novel_candidate,pp_even,fSC_AB,0.030153,"(0, 0)",0.0,0.0


In [6]:

for name, res in results.items():
    print("=" * 80)
    print(name)
    print(json.dumps(res.summary_dict(), indent=2))


PI
{
  "kernel": "ph_charge",
  "Q": [
    0.0,
    0.0
  ],
  "coarse_label": "d",
  "coarse_score": 1.0000000000000002,
  "paper_label": "PI",
  "paper_score": 0.33026732487248656,
  "recognition_status": "recognized",
  "top_template_name": "PI_dx2y2",
  "top_template_score": 0.33026732487248656,
  "spin_sector": "charge/singlet",
  "channel_sector": "ph",
  "degeneracy": 2,
  "internal_mode": {
    "channel_sector": "ph",
    "spin_sector": "charge/singlet",
    "tensor_basis_shape": [
      24,
      3,
      3,
      2
    ],
    "dominant_pair": [
      0,
      0
    ],
    "same_sublattice_weight": 1.0,
    "inter_sublattice_weight": 0.0,
    "pair_weights": [
      [
        1.0,
        0.0,
        0.0
      ],
      [
        0.0,
        0.0,
        0.0
      ],
      [
        0.0,
        0.0,
        0.0
      ]
    ]
  },
  "template_scores": [
    {
      "name": "PI_dx2y2",
      "score": 0.33026732487248656
    },
    {
      "name": "PI_dxy",
      "score": 0.330


## Expected physical interpretation

- **PI** should be recognized as `PI` and be diagonal / same-sublattice dominated.
- **cBO** should be recognized as `cBO` and be off-diagonal B–C dominated.
- **sBO** should be recognized as `sBO` and be off-diagonal B–C dominated.
- **fSC** should be recognized as `f-SC` and be off-diagonal A–B dominated.
- **FM** should be recognized as `FM`.
- **novel_pp_dwave** should *not* be forced into the five paper labels, and should remain `unclassified`.


In [7]:

assert results["PI"].paper_label == "PI"
assert results["PI"].internal_mode.dominant_pair == (0, 0)
assert abs(results["PI"].internal_mode.same_sublattice_weight - 1.0) < 1e-12

assert results["cBO"].paper_label == "cBO"
assert results["cBO"].internal_mode.dominant_pair == (1, 2)
assert abs(results["cBO"].internal_mode.inter_sublattice_weight - 1.0) < 1e-12

assert results["sBO"].paper_label == "sBO"
assert results["sBO"].internal_mode.dominant_pair == (1, 2)

assert results["fSC"].paper_label == "f-SC"
assert results["fSC"].internal_mode.dominant_pair == (0, 1)
assert abs(results["fSC"].internal_mode.inter_sublattice_weight - 1.0) < 1e-12

assert results["FM"].paper_label == "FM"

assert results["novel_pp_dwave"].paper_label == "unclassified"
assert results["novel_pp_dwave"].recognition_status == "novel_candidate"

print("All benchmark assertions passed.")


All benchmark assertions passed.
